In [1]:
import pandas as pd
import pygwalker as pyg
import os
import openpyxl

# https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/68bede3a-b80c-8333-8654-72449668a41a

In [2]:
chunk_count = 18832
intervals = 9
gross_outputs = 10
net_outputs = 6

In [6]:
chunk_stats_folder = '/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/'
core_veg_model_parquet_folder = f'{chunk_stats_folder}parquet_20250921_17_33_57__model_v1_0_0__2016_2024/'
summative_veg_model_parquet_folder = f'{chunk_stats_folder}parquet_20250924_14_32_06__summative_outputs_model_v1_0_0__2016_2024/'

In [7]:
gross_df_full = pd.read_parquet(f'{core_veg_model_parquet_folder}LULUCF_fluxes_20250921_17_33_45__v1_0_0__gross_outputs_1x1.parquet')
gross_df_full.head()

,chunk_id,tile_id,layer_name,pattern,years,chunk_name,tile_name,in_out,min_value,mean_value,max_value,count_value,sum_value,data_type,iso
0,-28_-60_-27_-59,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2015_2016,gross_emissions__AGC__MgCO2_ha_yr,2015_2016,50S_030W__-28_-60_-27_-59__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
1,-27_-60_-26_-59,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2015_2016,gross_emissions__AGC__MgCO2_ha_yr,2015_2016,50S_030W__-27_-60_-26_-59__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
2,177_-60_178_-59,50S_170E,gross_emissions__AGC__MgCO2_ha_yr_2015_2016,gross_emissions__AGC__MgCO2_ha_yr,2015_2016,50S_170E__177_-60_178_-59__gross_emissions__AG...,50S_170E__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,ATA
3,-27_-59_-26_-58,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2015_2016,gross_emissions__AGC__MgCO2_ha_yr,2015_2016,50S_030W__-27_-59_-26_-58__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
4,-27_-58_-26_-57,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2015_2016,gross_emissions__AGC__MgCO2_ha_yr,2015_2016,50S_030W__-27_-58_-26_-57__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS


In [8]:
# Classify each row as tropical (30N to 30S) or non-tropical

def is_tropical(chunk_id):
    try:
        parts = chunk_id.split('_')
        second_number = int(parts[1])
        return -30 <= second_number <= 29
    except:
        return False  # Handle bad formatting or missing data

gross_df_full['tropical'] = gross_df_full['chunk_id'].apply(is_tropical)
# gross_df_full[gross_df_full['tropical'] == True]

In [9]:
aggregated_df = gross_df_full.groupby(['pattern', 'years', 'iso', 'tropical'], as_index=False)['sum_value'].sum()
# aggregated_df

In [10]:
# Exports to a csv so data can be used in Excel or reused
aggregated_df.to_csv('global_aggregated_1_0_0.csv', index=False)

gross_df_full_wide = gross_df_full.pivot(
    index=['chunk_id', 'years', 'iso'],
    columns='pattern',
    values='sum_value'
).reset_index()
gross_df_full_wide
gross_df_full_wide.to_csv('global_v1_0_0_wide.csv', index=False)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_caOXtjID"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"dragId":"gw_2OV7","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_1VKz","name":"Gross (global)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_fYNyDVrv"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_0Ky8","name":"Net (global)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_FW9k84nj"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"dragId":"gw_Zzu_","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_vZXc","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension","rule":{"type":"not in","value":[]}},{"dragId":"gw_H2xo","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["CAN","CHN","IDN","USA","BRA","COD","RUS"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_yiIa","name":"Net (country)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_FW9k84nj"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"gw_QYC2"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_vZXc","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension","rule":{"type":"not in","value":[]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_B_S6","name":"Net (tropics vs. non-tropics)"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"view","query":[{"op":"aggregate","groupBy":["years","pattern"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"view","query":[{"op":"aggregate","groupBy":["years"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"years","rule":{"type":"not in","value":[]}},{"fid":"iso","rule":{"type":"one of","value":["CAN","CHN","IDN","USA","BRA","COD","RUS"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["years","iso"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"years","rule":{"type":"not in","value":[]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["years","tropical"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(aggregated_df, spec=vis_spec)

Analysis of single country data

In [ ]:
gross_df_example_ISO = gross_df_full[gross_df_full['iso'] == 'RUS']
# gross_df_RUS

In [ ]:
gross_df_example_chunk = gross_df_full[gross_df_full['chunk_id'] == '124_-25_125_-24']

In [ ]:
pyg.walk(gross_df_example_ISO, spec=vis_spec)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_caOXtjID"},{"fid":"chunk_id","name":"chunk_id","semanticType":"nominal","analyticType":"dimension","basename":"chunk_id","dragId":"GW_0jirmkBQ"},{"fid":"tile_id","name":"tile_id","semanticType":"nominal","analyticType":"dimension","basename":"tile_id","dragId":"GW_j04AvcwB"},{"fid":"layer_name","name":"layer_name","semanticType":"nominal","analyticType":"dimension","basename":"layer_name","dragId":"GW_h5qCiTVm"},{"fid":"chunk_name","name":"chunk_name","semanticType":"nominal","analyticType":"dimension","basename":"chunk_name","dragId":"GW_ElTfRjBW"},{"fid":"tile_name","name":"tile_name","semanticType":"nominal","analyticType":"dimension","basename":"tile_name","dragId":"GW_Dm3QzHcy"},{"fid":"in_out","name":"in_out","semanticType":"nominal","analyticType":"dimension","basename":"in_out","dragId":"GW_JOo63mt9"},{"fid":"data_type","name":"data_type","semanticType":"nominal","analyticType":"dimension","basename":"data_type","dragId":"GW_sg1nBtkh"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"min_value","name":"min_value","semanticType":"nominal","analyticType":"measure","basename":"min_value","dragId":"GW_fbkAJ5QE"},{"fid":"mean_value","name":"mean_value","semanticType":"quantitative","analyticType":"measure","basename":"mean_value","dragId":"GW_En57Sb4z"},{"fid":"max_value","name":"max_value","semanticType":"quantitative","analyticType":"measure","basename":"max_value","dragId":"GW_Ylbqc07Q"},{"fid":"count_value","name":"count_value","semanticType":"quantitative","analyticType":"measure","basename":"count_value","dragId":"GW_Bn9sR7mz"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"dragId":"gw_2OV7","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_1VKz","name":"Gross (Russia)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_fYNyDVrv"},{"fid":"chunk_id","name":"chunk_id","semanticType":"nominal","analyticType":"dimension","basename":"chunk_id","dragId":"GW_AiQSV4oD"},{"fid":"tile_id","name":"tile_id","semanticType":"nominal","analyticType":"dimension","basename":"tile_id","dragId":"GW_vW8RftCm"},{"fid":"layer_name","name":"layer_name","semanticType":"nominal","analyticType":"dimension","basename":"layer_name","dragId":"GW_gXUBV7oI"},{"fid":"chunk_name","name":"chunk_name","semanticType":"nominal","analyticType":"dimension","basename":"chunk_name","dragId":"GW_uOU7ySkd"},{"fid":"tile_name","name":"tile_name","semanticType":"nominal","analyticType":"dimension","basename":"tile_name","dragId":"GW_YhMk3dlX"},{"fid":"in_out","name":"in_out","semanticType":"nominal","analyticType":"dimension","basename":"in_out","dragId":"GW_Y5N0ypoM"},{"fid":"data_type","name":"data_type","semanticType":"nominal","analyticType":"dimension","basename":"data_type","dragId":"GW_4O0CmRBt"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"min_value","name":"min_value","semanticType":"nominal","analyticType":"measure","basename":"min_value","dragId":"GW_HWSgAqI1"},{"fid":"mean_value","name":"mean_value","semanticType":"quantitative","analyticType":"measure","basename":"mean_value","dragId":"GW_lqAbkP1L"},{"fid":"max_value","name":"max_value","semanticType":"quantitative","analyticType":"measure","basename":"max_value","dragId":"GW_XNpTcIjm"},{"fid":"count_value","name":"count_value","semanticType":"quantitative","analyticType":"measure","basename":"count_value","dragId":"GW_9gAZWGRw"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_0Ky8","name":"Net (Russia)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_fYNyDVrv"},{"fid":"chunk_id","name":"chunk_id","semanticType":"nominal","analyticType":"dimension","basename":"chunk_id","dragId":"GW_AiQSV4oD"},{"fid":"tile_id","name":"tile_id","semanticType":"nominal","analyticType":"dimension","basename":"tile_id","dragId":"GW_vW8RftCm"},{"fid":"layer_name","name":"layer_name","semanticType":"nominal","analyticType":"dimension","basename":"layer_name","dragId":"GW_gXUBV7oI"},{"fid":"chunk_name","name":"chunk_name","semanticType":"nominal","analyticType":"dimension","basename":"chunk_name","dragId":"GW_uOU7ySkd"},{"fid":"tile_name","name":"tile_name","semanticType":"nominal","analyticType":"dimension","basename":"tile_name","dragId":"GW_YhMk3dlX"},{"fid":"in_out","name":"in_out","semanticType":"nominal","analyticType":"dimension","basename":"in_out","dragId":"GW_Y5N0ypoM"},{"fid":"data_type","name":"data_type","semanticType":"nominal","analyticType":"dimension","basename":"data_type","dragId":"GW_4O0CmRBt"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"min_value","name":"min_value","semanticType":"nominal","analyticType":"measure","basename":"min_value","dragId":"GW_HWSgAqI1"},{"fid":"mean_value","name":"mean_value","semanticType":"quantitative","analyticType":"measure","basename":"mean_value","dragId":"GW_lqAbkP1L"},{"fid":"max_value","name":"max_value","semanticType":"quantitative","analyticType":"measure","basename":"max_value","dragId":"GW_XNpTcIjm"},{"fid":"count_value","name":"count_value","semanticType":"quantitative","analyticType":"measure","basename":"count_value","dragId":"GW_9gAZWGRw"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"fid":"chunk_id","name":"chunk_id","semanticType":"nominal","analyticType":"dimension","basename":"chunk_id","dragId":"gw_0t6P"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_3qTN","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["gross_emissions__AGC__MgCO2_ha_yr"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_Q8xg","name":"Gross (Russia by chunk)"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_j79c","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_agqQ","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_yIiQ","fid":"iso","name":"iso","basename":"iso","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"},{"fid":"tropical","name":"tropical","semanticType":"nominal","analyticType":"dimension","basename":"tropical","dragId":"GW_caOXtjID"},{"fid":"chunk_id","name":"chunk_id","semanticType":"nominal","analyticType":"dimension","basename":"chunk_id","dragId":"GW_0jirmkBQ"},{"fid":"tile_id","name":"tile_id","semanticType":"nominal","analyticType":"dimension","basename":"tile_id","dragId":"GW_j04AvcwB"},{"fid":"layer_name","name":"layer_name","semanticType":"nominal","analyticType":"dimension","basename":"layer_name","dragId":"GW_h5qCiTVm"},{"fid":"chunk_name","name":"chunk_name","semanticType":"nominal","analyticType":"dimension","basename":"chunk_name","dragId":"GW_ElTfRjBW"},{"fid":"tile_name","name":"tile_name","semanticType":"nominal","analyticType":"dimension","basename":"tile_name","dragId":"GW_Dm3QzHcy"},{"fid":"in_out","name":"in_out","semanticType":"nominal","analyticType":"dimension","basename":"in_out","dragId":"GW_JOo63mt9"},{"fid":"data_type","name":"data_type","semanticType":"nominal","analyticType":"dimension","basename":"data_type","dragId":"GW_sg1nBtkh"}],"measures":[{"dragId":"gw_Iqpt","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"fid":"min_value","name":"min_value","semanticType":"nominal","analyticType":"measure","basename":"min_value","dragId":"GW_fbkAJ5QE"},{"fid":"mean_value","name":"mean_value","semanticType":"quantitative","analyticType":"measure","basename":"mean_value","dragId":"GW_En57Sb4z"},{"fid":"max_value","name":"max_value","semanticType":"quantitative","analyticType":"measure","basename":"max_value","dragId":"GW_Ylbqc07Q"},{"fid":"count_value","name":"count_value","semanticType":"quantitative","analyticType":"measure","basename":"count_value","dragId":"GW_Bn9sR7mz"}],"rows":[{"dragId":"gw_UEfg","fid":"sum_value","name":"sum_value","basename":"sum_value","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_TKd_","fid":"years","name":"years","basename":"years","semanticType":"nominal","analyticType":"dimension"}],"color":[{"dragId":"gw_2OV7","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_0DOj","fid":"pattern","name":"pattern","basename":"pattern","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["gross_emissions__CH4__MgCO2e_ha_yr","gross_emissions__N2O__MgCO2e_ha_yr","gross_emissions__deadwood_C__MgCO2_ha_yr"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_pm26","name":"Gross non-CO2 (Russia)"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"view","query":[{"op":"aggregate","groupBy":["years","pattern"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"view","query":[{"op":"aggregate","groupBy":["years"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"pattern","rule":{"type":"one of","value":["gross_emissions__AGC__MgCO2_ha_yr"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["years","chunk_id"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"pattern","rule":{"type":"one of","value":["gross_emissions__CH4__MgCO2e_ha_yr","gross_emissions__N2O__MgCO2e_ha_yr","gross_emissions__deadwood_C__MgCO2_ha_yr"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["years","pattern"],"measures":[{"field":"sum_value","agg":"sum","asFieldKey":"sum_value_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(gross_df_example_ISO, spec=vis_spec)

# Core vegetation model parquet QC

In [ ]:
# Checks that there are the expected number of rows in the gross flux table
gross_df_rows = len(gross_df_full)
print(len(gross_df_full))
print(gross_df_rows == (chunk_count*intervals*gross_outputs))

In [ ]:
# Reads the dataframe that has global min and max values for all inputs and outputs
min_max_df = pd.read_parquet(f'{core_veg_model_parquet_folder}LULUCF_fluxes_20250921_17_33_45__v1_0_0__min_max_for_layers_1x1.parquet')
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.0f}'.format) 
# min_max_df

In [ ]:
# Checks that all outputs have the correct number of chunks
all_chunks_present = (min_max_df['count'] == chunk_count).all()
print((min_max_df['count'] == chunk_count).all() == True)

In [ ]:
# Functions to check min and max values for model inputs and outputs over various scales (chunk or global)
def check_min_max(df, min_max_dict):

    results = {}
    
    for key, (expected_min, expected_max) in min_max_dict.items():
        # Filter rows that match the string
        subset = df[df["layer_name"].str.contains(key)]
    
        # Apply the test
        condition = (subset["min_value"] <= expected_min) & (subset["max_value"] >= expected_max)
    
        # Save result
        results[key] = {
            "all_pass": condition.all(),
            "failing_rows": subset[~condition]  # rows that failed
        }

    return results


def show_min_max_results(extent, results):

    # Show results
    for key, res in results.items():
        print(f"  Min and max for {key} in {extent}: All rows pass--", res["all_pass"])
        if not res["all_pass"]:
            print("  Failing rows:")
            print(res["failing_rows"])

In [ ]:
# Checks min and max values for major sets of inputs and outputs (global-- across all chunks) to make sure they match expectations.
# That is, checks the global min and max, not chunk-level min and max. 
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant

# Input and output patterns and their expected min and max values
global_min_max_dict = {"AGC_emission_factor_CO2_only": [0, 1], "GMWv3_smoothed_mangrove_extent": [0, 1], "burned_area_final": [0, 1], 
                "carbon_density__AGC": [0, 400], "carbon_density__BGC": [0, 300], "carbon_density__deadwood_C": [0, 40], "carbon_density__litter_C": [0, 10],
                "composite_primary_forest": [0, 1], "forest_age_at_end_of_interval": [0, 300], "forest_age_gap_filled_start_year": [6, 305], "gain_year_count_during_interval": [0, 1],
                "gross_emissions__AGC__MgCO2_ha_yr": [0, 1300], "gross_emissions__BGC__MgCO2_ha_yr": [0, 800], "gross_emissions__CH4__MgCO2e_ha_yr": [0, 95], "gross_emissions__N2O__MgCO2e_ha_yr": [0, 25],
                "gross_emissions__deadwood_C__MgCO2_ha_yr": [0, 90], "gross_emissions__litter_C__MgCO2_ha_yr": [0, 40],
                "gross_removals__AGC__MgCO2_ha_yr": [-50, 0], "gross_removals__BGC__MgCO2_ha_yr": [-30, 0], 
                "gross_removals__deadwood_C__MgCO2_ha_yr": [-5, 0], "gross_removals__litter_C__MgCO2_ha_yr": [-0.5, 0],
                "land_cover_composite": [0, 255], "land_state_node": [10200000, 70000000], "max_height_since_last_time_not_tall_veg": [0, 66], "most_recent_year_not_tall_veg": [0, 2015],
                "partial_or_full_dist_in_current_interval": [0, 1], "removal_factor__AGC__MgC_ha_yr": [0, 15], "times_burned_in_current_interval": [0, 1], "vegetation_height": [0, 60]
               }

min_max_results = check_min_max(min_max_df, global_min_max_dict)

show_min_max_results("global", min_max_results)

In [ ]:
# Checks min and max values for major sets of inputs and outputs for specific chunks to make sure they match expectations.
# That is, checks chunk-level min and max.
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant

pd.set_option('display.float_format', '{:.4f}'.format) 

# Input and output patterns and their expected min and max values
gross_chunk_dict = { 
                "gross_emissions__AGC__MgCO2_ha_yr": [0, 100], "gross_emissions__BGC__MgCO2_ha_yr": [0, 50], 
                "gross_emissions__CH4__MgCO2e_ha_yr": [0, 10], "gross_emissions__N2O__MgCO2e_ha_yr": [0, 5],
                "gross_emissions__deadwood_C__MgCO2_ha_yr": [0, 3], "gross_emissions__litter_C__MgCO2_ha_yr": [0, 3],
                "gross_removals__AGC__MgCO2_ha_yr": [-10, 0], "gross_removals__BGC__MgCO2_ha_yr": [-10, 0], 
                "gross_removals__deadwood_C__MgCO2_ha_yr": [-0.002, 0], "gross_removals__litter_C__MgCO2_ha_yr": [-0.002, 0]
               }

# chunk_ids = ['24_-8_25_-7', '10_49_11_50']
chunk_ids = ['24_-8_25_-7']

for chunk_id in chunk_ids:
    print(f"Checking min and max in chunk {chunk_id}:")
    gross_df_chunk = gross_df_full[gross_df_full['chunk_id'] == chunk_id]
    
    min_max_results = check_min_max(gross_df_chunk, gross_chunk_dict)
    
    show_min_max_results(chunk_id, min_max_results)

# Summative vegetation model parquet QC

In [ ]:
# Reads the files that have net and summative gross chunk stats
net_df_full = pd.read_parquet(f'{summative_veg_model_parquet_folder}LULUCF_summative_output_calculation_20250924_14_32_04__v1_0_0__net_outputs_1x1.parquet')
summative_gross_df_full = pd.read_parquet(f'{summative_veg_model_parquet_folder}LULUCF_summative_output_calculation_20250924_14_32_04__v1_0_0__gross_outputs_1x1.parquet')

# Reads the file that has global min and max values for all inputs and outputs
min_max_df = pd.read_parquet(f'{summative_veg_model_parquet_folder}LULUCF_summative_output_calculation_20250924_14_32_04__v1_0_0__min_max_for_layers_1x1.parquet')
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.0f}'.format) 

# Dataframe of summative gross fluxes and net fluxes 
summative_df_full = pd.concat([net_df_full, summative_gross_df_full])

# summative_df_full.head()
# min_max_df.head()

In [ ]:
# Checks that there are the expected number of rows in the gross flux table
net_df_rows = len(net_df_full)
print(net_df_rows)
print(net_df_rows == (chunk_count*(intervals+1)*net_outputs))  # Add 1 to intervals because there table also includes the total across all intervals (2016-2024)

In [ ]:
# Checks that all outputs have the correct number of chunks
all_chunks_present = (min_max_df['count'] == chunk_count).all()
print((min_max_df['count'] == chunk_count).all() == True)

In [ ]:
# Checks min and max values for net flux outputs (across all chunks) to make sure they match expectations.
# That is, checks the global min and max, not chunk-level min and max. 
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant

# Input and output patterns and their expected min and max values
global_min_max_dict = {"carbon_density__non_soil": [0, 600], 
                "gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr": [0, 2000], "gross_emissions__all_C_pools__all_gases__MgCO2_ha_yr": [0, 2000], 
                "gross_emissions__all_C_pools__non_CO2_only__MgCO2e_ha_yr": [0, 100], 
                "gross_removals__all_C_pools__MgCO2_ha_yr": [-50, 0], 
                "net_flux__AGC__MgCO2": [-50, 1200], "net_flux__BGC__MgCO2": [-20, 700], "net_flux__all_C_pools__CO2_only__MgCO2": [-50, 2000], "net_flux__all_C_pools__all_gases__MgCO2": [-50, 2000],
                "net_flux__deadwood_C__MgCO2": [-5, 90], "net_flux__litter_C__MgCO2": [-0.5, 40]
               }

min_max_results = check_min_max(min_max_df, global_min_max_dict)

show_min_max_results("global", min_max_results)

In [ ]:
# Checks min and max values for summative gross and net flux outputs for specific chunks to make sure they match expectations.
# That is, checks chunk-level min and max. 
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant

pd.set_option('display.float_format', '{:.4f}'.format) 

# Input and output patterns and their expected min and max values
chunk_min_max_dict = {"carbon_density__non_soil": [0, 600], 
                "gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr": [0, 700], "gross_emissions__all_C_pools__all_gases__MgCO2_ha_yr": [0, 700], 
                "gross_emissions__all_C_pools__non_CO2_only__MgCO2e_ha_yr": [0, 30], 
                "gross_removals__all_C_pools__MgCO2_ha_yr": [-10, 0],
                "net_flux__AGC__MgCO2": [-10, 300], "net_flux__BGC__MgCO2": [-10, 100], 
                "net_flux__all_C_pools__CO2_only__MgCO2": [-10, 400], "net_flux__all_C_pools__all_gases__MgCO2": [-10, 400],
                "net_flux__deadwood_C__MgCO2": [-0.02, 4], "net_flux__litter_C__MgCO2": [-0.02, 2]
               }

# chunk_ids = ['24_-8_25_-7', '10_49_11_50']
chunk_ids = ['24_-8_25_-7']

for chunk_id in chunk_ids:
    print(f"Checking min and max in chunk {chunk_id}:")
    summative_df_chunk = summative_df_full[summative_df_full['chunk_id'] == chunk_id]

    min_max_results = check_min_max(summative_df_chunk, chunk_min_max_dict)
    
    show_min_max_results(chunk_id, min_max_results)
# summative_df_chunk

In [ ]:
%%time

# Compares select gross and corresponding net values for chunks and intervals
# Based on https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/68d561bb-9fdc-832b-9bb0-dd10c4da5c2c

years = gross_df_full['years'].unique()  # Years to compare
unique_chunk_ids = gross_df_full['chunk_id'].unique()  # List of all chunk_ids
unique_chunk_ids = pd.Series(unique_chunk_ids)
unique_chunk_ids = unique_chunk_ids.sample(n=10000, random_state=42).tolist()  # Random sample of chunk_ids to compare because 18832 will take about 4 minutes

# Gross vs. summative/net comparisons to conduct. 
# Key is the string to search the gross df patterns for (e.g., AGC emissions, BGC removals).
# Value is the corresponding net/summative value to compare.
# For example gross_emissions retrieves all six gross emissions (AGC, BGC, deadwood, litter, CH4, N2O) and compares with gross_emissions__all_C_pools__all_gases. 
gross_vs_summative_comparison_dict = {"gross_emissions": "gross_emissions__all_C_pools__all_gases", 
                                      "gross_removals": "gross_removals__all_C_pools", 
                                      "gross": "net_flux__all_C_pools__all_gases"}

# Iterates across comparisons
for key, value in gross_vs_summative_comparison_dict.items():
    print(f"=== Comparing {key} to {value} ===")

    # Iterates across years
    for year in years:
        print(f"  Comparing {year}")

        # Iterates across sample of chunk_ids
        for chunk_id in unique_chunk_ids:
            # print(f"    Comparing {chunk_id}")

            # Gross df filtered to the relevant values
            gross_filtered = gross_df_chunk[
                gross_df_chunk['pattern'].str.contains(key) &
                (gross_df_chunk['years'] == year) &
                (gross_df_chunk['chunk_id'] == chunk_id)
            ]
            # print(gross_filtered)

            # Summed gross value
            gross_sum_value = gross_filtered['sum_value'].sum()
            # print(gross_sum_value)
            
            # Gets corresponding value from summative table
            summative_filtered = summative_df_chunk[
                summative_df_chunk['pattern'].str.contains(value) &
                (summative_df_chunk['years'] == year) & 
                (summative_df_chunk['chunk_id'] == chunk_id)
            ]
            # summative_filtered
            
            corresponding_summative = summative_filtered['sum_value'].sum()
            # print(corresponding_summative)
        
            diff = gross_sum_value - corresponding_summative
            if abs(diff) < 4:  # The values won't be exactly the same because of floating point errors during summation, so there's a little tolerance here. 
                # print(f"      Gross {key} and summative {value} for {year} for {chunk_id}: match ")  # Not printing passes because it slows down the testing and is visually distracting. 
                pass
            else:
                print(f"      Gross {key} and summative {value} for {year} for {chunk_id}: do not match by {diff}")
                print(f"      gross filtered: {gross_sum_value}")
                print(gross_filtered)
                print("       corresponding_summative:")
                print(corresponding_summative)

# 0.04x0.04 deg chunk-level output vegetation model QC

In [ ]:
min_max_4km = pd.read_excel(f'{chunk_stats_folder}LULUCF_0_04deg_output_by_chunk_1x1_chunk_statistics_20250925_12_20_23__KEEP.xlsx', sheet_name="min_max_for_layers_1x1")
chunk_gross_4km = pd.read_excel(f'{chunk_stats_folder}LULUCF_0_04deg_output_by_chunk_1x1_chunk_statistics_20250925_12_20_23__KEEP.xlsx', sheet_name="gross_outputs_1x1")
# chunk_net_4km = pd.read_excel(f'{chunk_stats_folder}LULUCF_0_04deg_output_by_chunk_1x1_chunk_statistics_20250925_12_20_23__KEEP.xlsx', sheet_name="net_outputs_1x1")
min_max_4km.head()